# **Context Management**

### **What's Covered?**
1. Runtime Message Placeholder
    - How it Works?
    - Making MessagesPlaceholder Optional
    - Limiting the number of messages
2. State & Context Optimization
    - Trim Messages Based on Number of Tokens
    - Trim Messages Based on Number of Messages
    - Filter Messages

## **Runtime Message Placeholder**

At its core, `MessagesPlaceholder` is a special type of "placeholder" within a `ChatPromptTemplate` that is designed to accept a sequence of messages (like HumanMessage, AIMessage, SystemMessage).

### **How it Works?**  
When you include MessagesPlaceholder in your ChatPromptTemplate.from_messages() or ChatPromptTemplate() definition:
1. You give it a variable_name (e.g., "chat_history").
2. When you invoke() the ChatPromptTemplate (or a chain containing it), you must pass a key in your input dictionary that matches this variable_name. The value for this key must be a list of BaseMessage objects.
3. LangChain then takes this list of messages and inserts them directly into the position specified by MessagesPlaceholder within the final list of messages sent to the LLM.

In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Simulate some pre-existing chat history
current_chat_history = [
    HumanMessage(content="What's your favorite color?"),
    AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    HumanMessage(content="Oh, I see. What's your favorite animal then?"),
    AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals."),
]

# Simulate some pre-existing chat history
current_chat_history = [
    ("human", "What's your favorite color?"),
    ("ai", "As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    ("human", "Oh, I see. What's your favorite animal then?"),
    ("ai", "Similarly, I don't have personal experiences to develop preferences for animals."),
]

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate

# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate(
    messages=[
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history"),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ]
)

chat_prompt_with_history.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{new_question}


In [3]:
try:
    chat_prompt_with_history.invoke({"new_question": "Can you summarize our conversation so far?"})
except:
    print("KeyError: Input to ChatPromptTemplate is missing variables {'chat_history'}.")

KeyError: Input to ChatPromptTemplate is missing variables {'chat_history'}.


In [4]:
formatted_messages = chat_prompt_with_history.invoke(
    {
        "new_question": "Can you summarize our conversation so far?", 
        "chat_history": current_chat_history
    }
)

formatted_messages.to_messages()

[SystemMessage(content='You are a helpful and knowledgeable assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's your favorite color?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="Oh, I see. What's your favorite animal then?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you summarize our conversation so far?', additional_kwargs={}, response_metadata={})]

In [5]:
for msg in formatted_messages.to_messages():
    msg.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.
================================ Human Message =================================

What's your favorite color?
================================== Ai Message ==================================

As an AI, I don't have feelings or preferences, so I don't have a favorite color.
================================ Human Message =================================

Oh, I see. What's your favorite animal then?
================================== Ai Message ==================================

Similarly, I don't have personal experiences to develop preferences for animals.
================================ Human Message =================================

Can you summarize our conversation so far?


### **Making MessagesPlaceholder Optional**

In [6]:
# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ]
)

chat_prompt_with_history.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{new_question}


### **Limiting the number of messages**

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Simulate some pre-existing chat history
current_chat_history = [
    HumanMessage(content="What's your favorite color?"),
    AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    HumanMessage(content="Oh, I see. What's your favorite animal then?"),
    AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals."),
]

# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate(
    messages = [
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history", optional=True, n_messages=2),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ] 
)

formatted_messages = chat_prompt_with_history.invoke(
    {
        "chat_history": current_chat_history,
        "new_question": "Can you summarize our conversation so far?"
    }
)

for msg in formatted_messages.to_messages():
    msg.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.
================================ Human Message =================================

Oh, I see. What's your favorite animal then?
================================== Ai Message ==================================

Similarly, I don't have personal experiences to develop preferences for animals.
================================ Human Message =================================

Can you summarize our conversation so far?


## **State & Context Optimization**

### **Trim Messages Based on Number of Tokens**

Trim messages to be below a token count.

`trim_messages` can be used to reduce the size of a chat history to a specified token or message count.
 
Args:
- **strategy='last'**: Defines the strategy for trimming. `'first'`: Keep the first <= n_count tokens of the messages. `'last'`: Keep the last <= n_count tokens of the messages.
- **include_system=True**: Usually, the new chat history should include the SystemMessage if it was present in the original chat history since the SystemMessage includes special instructions to the chat model. The SystemMessage is almost always the first message in the history if present. To achieve this set the `include_system=True`. Should only be specified if `strategy="last"`.
- **start_on='human'**: The resulting chat history should be valid. Most chat models expect that chat history starts with either (1) a HumanMessage or (2) a SystemMessage followed by a HumanMessage. To achieve this, set `start_on='human'`. In addition, generally a ToolMessage can only appear after an AIMessage that involved a tool call. Does not apply to a SystemMessage at index 0 if `include_system=True.`


In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Simulate some pre-existing chat history
current_chat_history = [
    SystemMessage(content="You are a helpful AI assistant. Your name is John Doe."),
    HumanMessage(content="What's your favorite color?"),
    AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    HumanMessage(content="Oh, I see. What's your favorite animal then?"),
    AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals."),
]

# Simulate some pre-existing chat history
current_chat_history = [
    ("system", "You are a helpful AI assistant. Your name is John Doe."),
    ("human", "What's your favorite color?"),
    ("ai", "As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    ("human", "Oh, I see. What's your favorite animal then?"),
    ("ai", "Similarly, I don't have personal experiences to develop preferences for animals."),
]

In [9]:
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately

trimmed_messages = trim_messages(
    current_chat_history,
    strategy="last",
    token_counter=count_tokens_approximately,
    max_tokens=45,
    include_system=True,
)

trimmed_messages

[SystemMessage(content='You are a helpful AI assistant. Your name is John Doe.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### **Trim Messages Based on Number of Messages**

- **token_counter=len**: Passing in `len` as a token counter function will count the number of messages in the chat history.
- **max_tokens=4**: When `len` is passed in as the token counter function, max_tokens will count the number of messages in the chat history.

In [10]:
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately

trimmed_messages = trim_messages(
    current_chat_history,
    strategy="last",
    token_counter=len,
    max_tokens=3,
    include_system=True,
)

trimmed_messages

[SystemMessage(content='You are a helpful AI assistant. Your name is John Doe.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Oh, I see. What's your favorite animal then?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### **Filter Messages**

Filter messages based on type.

Args:
- **include_types:** Message types to include. Can be specified as string names (e.g. 'system', 'human', 'ai', ...) or as BaseMessage classes (e.g. SystemMessage, HumanMessage, AIMessage, ...).
- **exclude_types:** Message types to exclude. Can be specified as string names (e.g. 'system', 'human', 'ai', ...) or as BaseMessage classes (e.g. SystemMessage, HumanMessage, AIMessage, ...).
- **exclude_tool_calls:** Tool call IDs to exclude. Can be one of the following:
    - `True`: All `AIMessage` objects with tool calls and all `ToolMessage` objects will be excluded.
    - `a sequence of tool call IDs to exclude`: ToolMessage objects with the corresponding tool call ID will be excluded

In [11]:
from langchain_core.messages import filter_messages

filtered_messages = filter_messages(
    current_chat_history,
    include_types=("system", "human"),
)

filtered_messages

[SystemMessage(content='You are a helpful AI assistant. Your name is John Doe.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's your favorite color?", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Oh, I see. What's your favorite animal then?", additional_kwargs={}, response_metadata={})]

In [12]:
from langchain_core.messages import filter_messages

filtered_messages = filter_messages(
    current_chat_history,
    exclude_types=("ai"),
)

filtered_messages

[SystemMessage(content='You are a helpful AI assistant. Your name is John Doe.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's your favorite color?", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Oh, I see. What's your favorite animal then?", additional_kwargs={}, response_metadata={})]